In [ ]:
import os
import random
import subprocess
from math import ceil
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

from utils import (
    RESULT_DIR,
    SHELL,
    activate_env_cmd,
)

In [ ]:
results_abc = RESULT_DIR / "ABC_atlas"
processed_h5ad = results_abc / "Zhuang-ABCA-1.h5ad"

script_dir = Path("clustering")

seed = 42

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
out_dir = results_abc / "scalability"

out_dir.mkdir(exist_ok=True, parents=True)

# Downsample

In [ ]:
batch_key = "brain_section_label"

random.seed(seed)

adata = ad.read_h5ad(processed_h5ad)

batches = list(adata.obs[batch_key].cat.categories)
n_batches = len(batches)

n_cells = []

for i in range(4):
    fraction = 1 / 5**i
    n = int(ceil(n_batches * fraction))
    keep_batch = random.sample(batches, n)
    adata_subset = adata[adata.obs[batch_key].isin(keep_batch), :].copy()
    sc.pp.neighbors(adata_subset, use_rep="X_pca", random_state=seed)
    # remove unnecessary values to reduce file size and io
    del (
        adata_subset.var,
        adata_subset.varm,
        adata_subset.obsm["X_pca"],
        adata_subset.X,
        adata_subset.layers,
    )

    adata_subset.write_h5ad(out_dir / f"subset_{i}.h5ad")
    n_cells.append((i, adata_subset.shape[0], n))

n_cells = pd.DataFrame(n_cells, columns=["subsampling", "n_cells", "n_sections"])

del adata, adata_subset

# Benchmark

In [ ]:
conda_cmd = activate_env_cmd("spatialleiden_benchmark")

script_path = script_dir / "spatialleiden_benchmark.py"

time_cmd = "~/opt/bin/time --format='%e %M'"

In [ ]:
perf_metrics = []
for file in out_dir.glob("*h5ad"):
    print(f"Processing: {file}")
    for s in range(3):
        print(f"  seed: {s}")

        cmd = f"{script_path} {file} --weight 2.5 --seed {s} --directed"
        out = subprocess.run(
            f"{conda_cmd} && {time_cmd} {cmd}",
            shell=True,
            executable=SHELL,
            env=env,
            check=True,
            capture_output=True,
            text=True,
        )
        # time writes to stderr
        # time in seconds and memory in kilobytes
        time, mem = out.stderr.strip().split(" ")
        perf_metrics.append((file, s, float(time), int(mem)))

perf_metrics = pd.DataFrame(perf_metrics, columns=["file", "seed", "time", "mem"])

In [ ]:
column_w_unit = {"time": "time [s]", "mem": "memory (max RSS) [KB]"}

In [ ]:
metrics = perf_metrics.assign(
    subsampling=lambda df: df["file"].apply(lambda x: int(x.stem.split("_")[1]))
).join(n_cells.set_index("subsampling"), on="subsampling")

metrics = (
    metrics[["n_sections", "n_cells", "seed", "time", "mem"]]
    .sort_values(["n_sections", "seed"])
    .rename(columns=column_w_unit)
)

In [ ]:
metrics.to_csv(out_dir / "benchmark.tsv", sep="\t", index=False)

# Plotting

In [ ]:
metrics = pd.read_table(out_dir / "benchmark.tsv")

In [ ]:
metrics = metrics.assign(
    **{"memory [MB]": lambda df: df["memory (max RSS) [KB]"] / 1_000}
)

In [ ]:
scatter_kwargs = dict(
    lw=0, alpha=0.5, style="seed", hue="seed", palette="tab10", legend=False
)

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(3, 4))
sns.scatterplot(data=metrics, x="n_cells", y="time [s]", ax=axs[0], **scatter_kwargs)
axs[0].set(ylim=(1, 1e4), yscale="log")

sns.scatterplot(data=metrics, x="n_cells", y="memory [MB]", ax=axs[1], **scatter_kwargs)
axs[1].set(ylim=(1e2, 1e5), yscale="log")
axs[1].set(xscale="log", xlabel=r"$\rm n_{cells}$", xlim=(1e4, 1e7))

fig.tight_layout()